In [26]:
from dotenv import load_dotenv
load_dotenv()

import os

import numpy as np
from pixell import enmap, enplot, reproject
import glob
import matplotlib.pyplot as plt
import emcee, corner
from astropy.io import fits
import sys
from astropy import units as u, constants as const
sys.path.insert(0, '../src')
#sys.path.insert(0, "/home/gill/apps/szpack/python")
import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
import yaml
import itertools
from pixell import enmap

# Compton y calculation
from astropy.constants import k_B, m_p, c, sigma_T, m_e
from astropy import units as u

from astropy.coordinates import SkyCoord
import astropy.units as u

import utils as ut

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
def calc_y_and_err(tau, tau_err, Te, Te_err):
    Te = Te * u.keV
    Te_err = Te_err * u.keV
    Te_K = (Te / k_B).to(u.K)    
    Te_err_K = (Te_err / k_B).to(u.K)

    y = tau * k_B * Te_K / (m_e * c**2).to(u.J)
    yerr = y * np.sqrt((tau_err / tau)**2 + (Te_err_K / Te_K)**2)

    return y.value, yerr.value

def calc_mean_and_std(samples, idx):
    data = samples[:, idx]

    mcmc_run = np.percentile(data, [16, 50, 84])
    err = .5 * (mcmc_run[2] - mcmc_run[0])
    
    # Calculate bin width using Freedman-Diaconis rule
    n = len(data)
    iqr = np.percentile(data, 75) - np.percentile(data, 25)
    bin_width = 2 * iqr * n**(-1/3)

    # Calculate number of bins using bin width
    num_bins = int((np.max(data) - np.min(data)) / bin_width)

    # Create histogram
    hist, bin_edges = np.histogram(data, bins=num_bins, density=True)

    # Find bin with maximum value (estimate of mode)
    mode_bin = np.argmax(hist)
    mode_value = 0.5 * (bin_edges[mode_bin] + bin_edges[mode_bin + 1])

    mean = mode_value
    std = err
    return mean, std

In [28]:
# indices for the chain parameters
param_indices = {
    'ra_a401': 0,
    'dec_a401': 1,
    'beta_a401': 2,
    'rc_a401': 3,
    'R_a401': 4,
    'theta_a401': 5,
    'tau_a401': 6,
    'Te_a401': 7,
    'AD_a401': 8,
    'ra_a399': 9,
    'dec_a399': 10,
    'beta_a399': 11,
    'rc_a399': 12,
    'R_a399': 13,
    'theta_a399': 14,
    'tau_a399': 15,
    'Te_a399': 16,
    'AD_a399': 17,
    'fil_ra': 18,
    'fil_dec': 19,
    'fil_l0': 20,
    'fil_w0': 21,
    'tau_fil': 22,
    'Te_fil': 23,
    'fil_AD': 24,
    'fil_vavg': 25
}

case10 = f"/home/gill/research/ACT/bridge/results/ajay/final_runs/case24/chain.h5"
case24 = f"/home/gill/research/ACT/bridge/results/ajay/final_runs/case24/chain.h5"
case25 = f"/home/gill/research/ACT/bridge/results/ajay/final_runs/case25/chain.h5"
case26 = f"/home/gill/research/ACT/bridge/results/ajay/final_runs/case26/chain.h5"
case27 = f"/home/gill/research/ACT/bridge/results/ajay/final_runs/case27/chain.h5"

In [29]:
burnin = 10000
thin = 1
samples_case10 = emcee.backends.HDFBackend(case10).get_chain(discard=burnin, flat=True, thin=thin)
samples_case24 = emcee.backends.HDFBackend(case24).get_chain(discard=burnin, flat=True, thin=thin)
samples_case25 = emcee.backends.HDFBackend(case25).get_chain(discard=burnin, flat=True, thin=thin)
samples_case26 = emcee.backends.HDFBackend(case26).get_chain(discard=burnin, flat=True, thin=thin)
samples_case27 = emcee.backends.HDFBackend(case27).get_chain(discard=burnin, flat=True, thin=thin)

In [30]:
# Compton y fits
# --- A401 ---
# Compton-y fit
compton_y_fits = {
    "a401_ra_mean": 44.751,
    "a401_ra_err": 0.002,
    "a401_dec_mean": 13.572,
    "a401_dec_err": 0.002,
    "a401_beta_mean": 0.82,
    "a401_beta_err": 0.05,
    "a401_rc_mean": 2.6,
    "a401_rc_err": 0.35,
    "a401_e_mean": 0.82,
    "a401_e_err": 0.055,
    "a401_theta_mean": 123,
    "a401_theta_err": 8.5,
    "a401_y_mean": 1.260e-04,
    "a401_y_err": 6.00e-06,
    
    "a399_ra_mean": 44.473,
    "a399_ra_err": 0.004,
    "a399_dec_mean": 13.03,
    "a399_dec_err": 0.003,
    "a399_beta_mean": 0.81,
    "a399_beta_err": 0.095,
    "a399_rc_mean": 3.0,
    "a399_rc_err": 0.65,
    "a399_e_mean": 0.93,
    "a399_e_err": 0.06,
    "a399_theta_mean": 133,
    "a399_theta_err": 26.5,
    "a399_y_mean": 8.100e-05,
    "a399_y_err": 6.00e-06,
    
    "fil_ra_mean": 44.68,
    "fil_ra_err": 0.02,
    "fil_dec_mean": 13.37,
    "fil_dec_err": 0.025,
    "fil_l_mean": 12.3,
    "fil_l_err": 1.55,
    "fil_w_mean": 10.8,
    "fil_w_err": 1.05,
    "fil_y_mean": 1.100e-05,
    "fil_y_err": 1.750e-06,
}


In [31]:
def result_comparer(samples, compton_y_fits, param_indices, cf_path):
    # create a dictionary of the sample results
    # create the labels first for the dictionary

    ## Take care of RA and DEC values 
    cf = ut.get_config_file(f'{cf_path}')

    region = ut.get_region(cf['region_center_ra'], 
                           cf['region_center_dec'], 
                           cf['region_width'])

    dire_data = "/home/gill/research/ACT/bridge/data_paper/data/data/act_no_reproj"
    data_ref = enmap.read_map(f"{dire_data}/act_cut_dr6v2_pa5_f098_4way_coadd_map_srcfree.fits", 
                              box=region)
    
    a401_ra_pix = samples[:, param_indices['ra_a401']]
    a401_dec_pix = samples[:, param_indices['dec_a401']]

    a399_ra_pix = samples[:, param_indices['ra_a399']]
    a399_dec_pix = samples[:, param_indices['dec_a399']]

    fil_ra_pix = samples[:, param_indices['fil_ra']]
    fil_dec_pix = samples[:, param_indices['fil_dec']]

    ra_a401, dec_a401 = data_ref.wcs.celestial.wcs_pix2world(a401_ra_pix, a401_dec_pix, 0)
    samples[:, param_indices['ra_a401']] = ra_a401
    samples[:, param_indices['dec_a401']] = dec_a401

    ra_a399, dec_a399 = data_ref.wcs.celestial.wcs_pix2world(a399_ra_pix, a399_dec_pix, 0)
    samples[:, param_indices['ra_a399']] = ra_a399
    samples[:, param_indices['dec_a399']] = dec_a399

    fil_ra, fil_dec = data_ref.wcs.celestial.wcs_pix2world(fil_ra_pix, fil_dec_pix, 0)
    samples[:, param_indices['fil_ra']] = fil_ra
    samples[:, param_indices['fil_dec']] = fil_dec
    
    # create a dictionary to hold the results
    labels = {
        "ra_a401_mean": 0,
        "ra_a401_std": 0,

        "dec_a401_mean": 0,
        "dec_a401_std": 0,

        "beta_a401_mean": 0,
        "beta_a401_std": 0,

        "rc_a401_mean": 0,
        "rc_a401_std": 0,

        "R_a401_mean": 0,
        "R_a401_std": 0,

        "theta_a401_mean": 0,
        "theta_a401_std": 0,

        "tau_a401_mean": 0,
        "tau_a401_std": 0,

        "Te_a401_mean": 0,
        "Te_a401_std": 0,

        "AD_a401_mean": 0,
        "AD_a401_std": 0,

        "ra_a399_mean": 0,
        "ra_a399_std": 0,

        "dec_a399_mean": 0,
        "dec_a399_std": 0,

        "beta_a399_mean": 0,
        "beta_a399_std": 0,

        "rc_a399_mean": 0,
        "rc_a399_std": 0,

        "R_a399_mean": 0,
        "R_a399_std": 0,

        "theta_a399_mean": 0,
        "theta_a399_std": 0,

        "tau_a399_mean": 0,
        "tau_a399_std": 0,

        "Te_a399_mean": 0,
        "Te_a399_std": 0,

        "AD_a399_mean": 0,
        "AD_a399_std": 0,


        "fil_ra_mean": 0,
        "fil_ra_std": 0,

        "fil_dec_mean": 0,
        "fil_dec_std": 0,

        "fil_l0_mean": 0,
        "fil_l0_std": 0,

        "fil_w0_mean": 0,
        "fil_w0_std": 0,

        "tau_fil_mean": 0,
        "tau_fil_std": 0,

        "Te_fil_mean": 0,
        "Te_fil_std": 0,

        "fil_AD_mean": 0,
        "fil_AD_std": 0,


        "fil_vavg_mean": 0,
        "fil_vavg_std": 0,

        "y_a401_mean": 0,
        "y_a401_std": 0,
                "y_a399_mean": 0,
        "y_a399_std": 0,

                "y_fil_mean": 0,
        "y_fil_std": 0,




    }

    # calculate the mean and std for each param in labels
    for key in labels.keys():
        if "std" in key:
            continue
        
        if "y_" in key:
            object_name = key.split('_')[1]
            Te = labels[f'Te_{object_name}_mean']
            Te_err = labels[f'Te_{object_name}_std']
            tau = labels[f'tau_{object_name}_mean']
            tau_err = labels[f'tau_{object_name}_std']
            y, yerr = calc_y_and_err(tau, tau_err, Te, Te_err)

            labels[key] = y
            labels[key.replace("mean", "std")] = yerr
        else:
            idx = param_indices[key.split('_')[0] + '_' + key.split('_')[1]]
            mean, std = calc_mean_and_std(samples, idx)
            labels[key] = mean
            labels[key.replace("mean", "std")] = std

    print("Results:")
    for key, value in labels.items():
        print(f"{key}: {value:.7f}")

In [32]:
config = f"/home/gill/research/ACT/multi-freq-bridge/configs/case10_ajay.yaml"

result_comparer(samples_case10, compton_y_fits, param_indices, cf_path=config)

Results:
ra_a401_mean: 44.6413671
ra_a401_std: 0.0016085
dec_a401_mean: 13.4806210
dec_a401_std: 0.0023115
beta_a401_mean: 1.2934148
beta_a401_std: 0.0979539
rc_a401_mean: 5.0170171
rc_a401_std: 0.4693173
R_a401_mean: 1.2482461
R_a401_std: 0.0806953
theta_a401_mean: 107.9655211
theta_a401_std: 6.2985047
tau_a401_mean: 0.0057957
tau_a401_std: 0.0003514
Te_a401_mean: 10.6180583
Te_a401_std: 0.2493504
AD_a401_mean: 143730.5921688
AD_a401_std: 206914.8259407
ra_a399_mean: 44.3649345
ra_a399_std: 0.0032139
dec_a399_mean: 12.9417210
dec_a399_std: 0.0036918
beta_a399_mean: 0.8671079
beta_a399_std: 0.0321705
rc_a399_mean: 4.7207761
rc_a399_std: 0.6200986
R_a399_mean: 1.4167152
R_a399_std: 0.1373508
theta_a399_mean: 126.7166111
theta_a399_std: 6.6449160
tau_a399_mean: 0.0048463
tau_a399_std: 0.0003318
Te_a399_mean: 9.0183831
Te_a399_std: 0.1897302
AD_a399_mean: 1281.9872710
AD_a399_std: 156592.2988437
fil_ra_mean: 44.5978648
fil_ra_std: 0.0232016
fil_dec_mean: 13.3132014
fil_dec_std: 0.0341491
